# **Introdução**

A imobiliária *InsightPlaces*, situada na cidade do Rio de Janeiro, está enfrentando dificuldades para alugar e vender imóveis. Em uma pesquisa de como empresas semelhantes operam no mercado, a *InsightPlaces* percebeu que esse problema pode estar relacionado aos valores dos imóveis e às recomendações que faz. Ela quer um modelo mais rápido, preciso na hora de precificar imóveis com base em suas características, e funcional para uma grande quantidade de volume de requisições recebidas. Não só isso com também precisa de um novo modelo de recomendação de imóveis, onde seu atual modelo recomenda apenas imóveis de mesma região de seu imóvel de interesse.

Nosso trabalho está em extrair e tratar seus dados e construir novos modelos para o time de corretores da *InsidesPlaces*. Portanto separaremos nosso trabalho em três semanas, com intuito de organizar e separar os trabalhos.

# **Semana 1**

Nesta Semana extrairemos o conjunto dos dados da empresa para fazer tratamento, faremos sua exploração e nosso primeiros tratamentos em suas colunas.

Para esta seman utilizaremos as seguintes bibliotecas

- Pyspark
- Requests
- Zipfile
- Os

Sendo **Pyspark** a principal biblioteca onde utilizaremos para extração, exploração e tratamento dos dados, enquanto as outras apenas para baixar nossos arquivos.

## **Extração dos Dados**

In [18]:
# import zipfile
# import os
# import requests

# # Caminho do arquivo ZIP
# zip_url = "https://caelum-online-public.s3.amazonaws.com/challenge-spark/semana-1.zip"
# zip_path = "./Semanas_zip/semana-1.zip"
# extract_path = "./Dataset"

# # Baixar o arquivo ZIP
# response = requests.get(zip_url)
# with open(zip_path, "wb") as f:
#     f.write(response.content)

# # Extrair o conteúdo
# with zipfile.ZipFile(zip_path, "r") as zip_ref:
#     zip_ref.extractall(extract_path)

# # Listar os arquivos extraídos
# os.listdir(extract_path)

In [19]:
from pyspark.sql import SparkSession

# Inicializar o Spark
spark = SparkSession.builder.appName("ProcessamentoJSON").getOrCreate()

# Caminho do arquivo JSON
json_path = "./Dataset/dataset_bruto.json"

# Carregar o arquivo JSON em um DataFrame
df = spark.read.json(json_path)

# Exibir as primeiras linhas
df.show(10)

+--------------------+--------------------+--------------------+
|             anuncio|             imagens|             usuario|
+--------------------+--------------------+--------------------+
|{0, [], [16], [0]...|[{39d6282a-71f3-4...|{9d44563d-3405-4e...|
|{0, [], [14], [0]...|[{23d2b3ab-45b0-4...|{36245be7-70fe-40...|
|{0, [1026], [1026...|[{1da65baa-368b-4...|{9dc415d8-1397-4d...|
|{0, [120], [120],...|[{79b542c6-49b4-4...|{9911a2df-f299-4a...|
|{0, [3], [3], [0]...|[{e2bc497b-6510-4...|{240a7aab-12e5-40...|
|{0, [20], [15], [...|[{2de09d46-dc0d-4...|{3c7057f5-0923-42...|
|{3, [43], [43], [...|[{147a80d9-cd40-4...|{5a9736b5-aaa0-4a...|
|{2, [42], [42], [...|[{35740004-063d-4...|{ec48d96a-137c-49...|
|{0, [], [12], [0]...|[{6d3d2aec-c96f-4...|{dad7db63-e19c-44...|
|{1, [41], [41], [...|[{3d404069-418e-4...|{a845f35f-3ab3-46...|
+--------------------+--------------------+--------------------+
only showing top 10 rows



Como nosso primeiro problema, nossos dados estão separados em 3 colunas diferentes.

Queremos a apenas utilizaremos os dados que estão nas coluna `anuncio`, aliás vamos ver qual é o tipo da coluna para conseguir extrair os dados que nele estão. Utilizaremos a função `printSchema()` para o trabalho.

In [20]:
df.printSchema()

root
 |-- anuncio: struct (nullable = true)
 |    |-- andar: long (nullable = true)
 |    |-- area_total: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- area_util: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- banheiros: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- caracteristicas: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- endereco: struct (nullable = true)
 |    |    |-- bairro: string (nullable = true)
 |    |    |-- cep: string (nullable = true)
 |    |    |-- cidade: string (nullable = true)
 |    |    |-- estado: string (nullable = true)
 |    |    |-- latitude: double (nullable = true)
 |    |    |-- longitude: double (nullable = true)
 |    |    |-- pais: string (nullable = true)
 |    |    |-- rua: string (nullable = true)
 |    |    |-- zona: string (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-

Vemos que ela na verdade é um `Struct`, para conseguirmos extraírmos nossos dados, utilizaremos de uma sintaxe especial do **SQL** para este caso.

In [21]:
df = df.select("*", "anuncio.*")
df = df.drop(*['anuncio', 'imagens', 'usuario'])
df.show(5)

+-----+----------+---------+---------+--------------------+--------------------+--------------------+-------+------+------------+------------+-----------+----+--------------------+
|andar|area_total|area_util|banheiros|     caracteristicas|            endereco|                  id|quartos|suites|tipo_anuncio|tipo_unidade|   tipo_uso|vaga|             valores|
+-----+----------+---------+---------+--------------------+--------------------+--------------------+-------+------+------------+------------+-----------+----+--------------------+
|    0|        []|     [16]|      [0]|                  []|{Centro, 20061003...|47d553e0-79f2-4a4...|    [0]|   [0]|       Usado|      Outros|  Comercial| [1]|[{260, 107, Venda...|
|    0|        []|     [14]|      [0]|                  []|{Centro, 20051040...|b6ffbae1-17f6-487...|    [0]|    []|       Usado|      Outros|  Comercial| [0]|[{260, 107, Venda...|
|    0|    [1026]|   [1026]|      [0]|                  []|{Maria da Graça, ...|1fb030a5-9e3e-4

In [22]:
df.select("tipo_anuncio").distinct().show()

+------------+
|tipo_anuncio|
+------------+
|       Usado|
|  Lançamento|
+------------+



## **Tratamento dos Dados**

### **Filtro**

O time de Data Science solicitou que fizéssemos alguns filtros nas colunas `tipo_uso`, `tipo_unidade` e `tipo_anuncio` da nossa base de dados:

- tipo_uso: **Residencial**;

- tipo_unidade: **Apartamento**;

- tipo_anuncio: **Usado**.

Mas ainda não saber o porquê. Vamos visualizar porque esse procedimento é importante antes de fazermos um filtro em nossa tabela.

Para nosso futuros tratamentos, segue a importante de `functions` de `pyspark.sql`.

In [23]:
from pyspark.sql import functions as f

In [24]:
for coluna in ['tipo_uso', 'tipo_unidade', 'tipo_anuncio']:
    df_freq = df.groupBy(coluna).count()

    total_linhas = df.count()

    df_freq = df_freq.withColumn(
        "frequencia_percentual", 
        f.round((f.col("count") / total_linhas) * 100, 2)
    )
    print(f'Tabela da coluna {coluna}:')
    df_freq.orderBy(f.desc("count")).show()
    print('========' * 6, '\n')

Tabela da coluna tipo_uso:


+-----------+-----+---------------------+
|   tipo_uso|count|frequencia_percentual|
+-----------+-----+---------------------+
|Residencial|84541|                 94.9|
|  Comercial| 4542|                  5.1|
+-----------+-----+---------------------+


Tabela da coluna tipo_unidade:


+------------+-----+---------------------+
|tipo_unidade|count|frequencia_percentual|
+------------+-----+---------------------+
| Apartamento|66801|                74.99|
|      Outros|11963|                13.43|
|        Casa|10319|                11.58|
+------------+-----+---------------------+




Tabela da coluna tipo_anuncio:


+------------+-----+---------------------+
|tipo_anuncio|count|frequencia_percentual|
+------------+-----+---------------------+
|       Usado|88827|                99.71|
|  Lançamento|  256|                 0.29|
+------------+-----+---------------------+




Vemos que algumas valores categóricos são apenas ruídos para nosso modelos.

Dessa forma seguiremos filtrando nosso DataFrame.

In [25]:
df_final = df.filter((df['tipo_uso'] == 'Residencial') \
                           & (df['tipo_unidade'] == 'Apartamento') \
                           & (df['tipo_anuncio'] == 'Usado'))

print(df_final.select('tipo_anuncio').distinct().show(1))
print(df_final.select('tipo_unidade').distinct().show(1))
print(df_final.select('tipo_anuncio').distinct().show(1))

+------------+
|tipo_anuncio|
+------------+
|       Usado|
+------------+

None


+------------+
|tipo_unidade|
+------------+
| Apartamento|
+------------+

None


+------------+
|tipo_anuncio|
+------------+
|       Usado|
+------------+

None


### **Tratamento de Tipos**

Seguindo com o tratamento dos dados, queremos ver quais colunas devemos tratar seu tipos de dados e, como vimos antes, há bastante erro em nossos dados.

Faremos uma visualização incial e aplicaremos algumas técnicas inciais para trocar o tipo.

In [26]:
df_final.printSchema()

root
 |-- andar: long (nullable = true)
 |-- area_total: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- area_util: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- banheiros: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- caracteristicas: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- endereco: struct (nullable = true)
 |    |-- bairro: string (nullable = true)
 |    |-- cep: string (nullable = true)
 |    |-- cidade: string (nullable = true)
 |    |-- estado: string (nullable = true)
 |    |-- latitude: double (nullable = true)
 |    |-- longitude: double (nullable = true)
 |    |-- pais: string (nullable = true)
 |    |-- rua: string (nullable = true)
 |    |-- zona: string (nullable = true)
 |-- id: string (nullable = true)
 |-- quartos: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- suites: array (nullable = true)
 |    |-- element: long (c

Como é percepitível, algumas colunas estão com estrutura de array e isso atrapalha se quisermos fazer um modelo de machine learning posteriormente. Nesse contexto, transformaremos os dados das colunas `quartos`, `suites`, `banheiros`, `vaga`, `area_total` e `area_util` de listas para inteiros.

Para isso vamos importar nosso tipos `IntegerType` de `pyspark.sql.types`.

In [27]:
from pyspark.sql.types import IntegerType

In [28]:
df_final = df_final.withColumn("quartos",df_final.quartos[0].cast(IntegerType()))
df_final = df_final.withColumn("suites",df_final.suites[0].cast(IntegerType()))
df_final = df_final.withColumn("banheiros",df_final.banheiros[0].cast(IntegerType()))
df_final = df_final.withColumn("vaga",df_final.vaga[0].cast(IntegerType()))
df_final = df_final.withColumn("area_total",df_final.area_total[0].cast(IntegerType()))
df_final = df_final.withColumn("area_util",df_final.area_util[0].cast(IntegerType()))

In [29]:
df_final.printSchema()

root
 |-- andar: long (nullable = true)
 |-- area_total: integer (nullable = true)
 |-- area_util: integer (nullable = true)
 |-- banheiros: integer (nullable = true)
 |-- caracteristicas: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- endereco: struct (nullable = true)
 |    |-- bairro: string (nullable = true)
 |    |-- cep: string (nullable = true)
 |    |-- cidade: string (nullable = true)
 |    |-- estado: string (nullable = true)
 |    |-- latitude: double (nullable = true)
 |    |-- longitude: double (nullable = true)
 |    |-- pais: string (nullable = true)
 |    |-- rua: string (nullable = true)
 |    |-- zona: string (nullable = true)
 |-- id: string (nullable = true)
 |-- quartos: integer (nullable = true)
 |-- suites: integer (nullable = true)
 |-- tipo_anuncio: string (nullable = true)
 |-- tipo_unidade: string (nullable = true)
 |-- tipo_uso: string (nullable = true)
 |-- vaga: integer (nullable = true)
 |-- valores: array (nullable = true)
 

In [30]:
df_final.show(5)

+-----+----------+---------+---------+--------------------+--------------------+--------------------+-------+------+------------+------------+-----------+----+--------------------+
|andar|area_total|area_util|banheiros|     caracteristicas|            endereco|                  id|quartos|suites|tipo_anuncio|tipo_unidade|   tipo_uso|vaga|             valores|
+-----+----------+---------+---------+--------------------+--------------------+--------------------+-------+------+------------+------------+-----------+----+--------------------+
|    3|        43|       43|        1|[Academia, Churra...|{Paciência, 23585...|d2e3a3aa-09b5-45a...|      2|  NULL|       Usado| Apartamento|Residencial|   1|[{245, NULL, Vend...|
|    2|        42|       42|        1|[Churrasqueira, P...|{Paciência, 23585...|085bab2c-87ad-452...|      2|  NULL|       Usado| Apartamento|Residencial|   1|[{0, 0, Venda, 15...|
|    1|        41|       41|        1|[Portaria 24h, Co...|{Guaratiba, 23036...|18d22cbe-1b86-4

E como podemos ver, nosso primeiro problema foi embora.

### **Tratamento de Colunas**

Como as colunas `endereco` e `valores` estão compactadas, extrairemos em DataFrames diferentes para mais tarde utilizarmos ela.

Faremos um procedimento análogo ao que fizemos no início.

In [31]:
df_final = df_final.select('*', 'endereco.*').drop('endereco')

df_final = df_final.withColumn('valores', f.explode('valores'))
df_final = df_final.select('*', 'valores.*').drop('valores')

df_final.show(5)

+-----+----------+---------+---------+--------------------+--------------------+-------+------+------------+------------+-----------+----+---------+--------+--------------+--------------+----------+----------+----+--------------------+----------+----------+----+-----+-----+
|andar|area_total|area_util|banheiros|     caracteristicas|                  id|quartos|suites|tipo_anuncio|tipo_unidade|   tipo_uso|vaga|   bairro|     cep|        cidade|        estado|  latitude| longitude|pais|                 rua|      zona|condominio|iptu| tipo|valor|
+-----+----------+---------+---------+--------------------+--------------------+-------+------+------------+------------+-----------+----+---------+--------+--------------+--------------+----------+----------+----+--------------------+----------+----------+----+-----+-----+
|    3|        43|       43|        1|[Academia, Churra...|d2e3a3aa-09b5-45a...|      2|  NULL|       Usado| Apartamento|Residencial|   1|Paciência|23585430|Rio de Janeiro|Rio

Feito mais esses ajustes, temos mais algumas coisas que precisamos fazer.

A equipe de ciência de dados nos solicitou que apenas as informações sobre bairro e zona da cidade fossem extraídas. Logo retiraremos as outras colunas.

Além do mais, a *InsightPlaces* permite que o(a) anunciante crie um anúncio com duas opções de valor. Assim, o(a) cliente pode criar um anúncio que mostre tanto o valor de venda do imóvel quanto o seu valor de locação, juntamente com os valores de taxa de condomínio (quando houver) e taxa de IPTU. Estes valores são diferenciados pelo campo tipo que pode assumir os valores Venda e Aluguel.

Como se trata de um estudo sobre o preço de venda dos imóveis, o time de cientistas de dados também solicitou apenas as informações do tipo `Venda`. Logo faremos um filtro a partir desta coluna.

In [32]:
df_final = df_final.drop('cep')
df_final = df_final.drop('cidade')
df_final = df_final.drop('estado')
df_final = df_final.drop('latitude')
df_final = df_final.drop('longitude')
df_final = df_final.drop('pais')
df_final = df_final.drop('rua')

In [33]:
df_final = df_final.filter(df_final['tipo'] == 'Venda')

df_final.select('tipo').distinct().show()

+-----+
| tipo|
+-----+
|Venda|
+-----+



In [34]:
df_final.show(5)

+-----+----------+---------+---------+--------------------+--------------------+-------+------+------------+------------+-----------+----+---------+----------+----------+----+-----+-----+
|andar|area_total|area_util|banheiros|     caracteristicas|                  id|quartos|suites|tipo_anuncio|tipo_unidade|   tipo_uso|vaga|   bairro|      zona|condominio|iptu| tipo|valor|
+-----+----------+---------+---------+--------------------+--------------------+-------+------+------------+------------+-----------+----+---------+----------+----------+----+-----+-----+
|    3|        43|       43|        1|[Academia, Churra...|d2e3a3aa-09b5-45a...|      2|  NULL|       Usado| Apartamento|Residencial|   1|Paciência|Zona Oeste|       245|NULL|Venda|15000|
|    2|        42|       42|        1|[Churrasqueira, P...|085bab2c-87ad-452...|      2|  NULL|       Usado| Apartamento|Residencial|   1|Paciência|Zona Oeste|         0|   0|Venda|15000|
|    1|        41|       41|        1|[Portaria 24h, Co...|1

## **Salvando os Dados**

Por fim salvaremos o arquivo em formato parquet e csv e compararemos os dois. 

Como csv não suporta o modelo de estrutura da coluna `características`, para este caso transformaremos em uma `string`.

In [38]:
df_final.write.parquet('./Salvamentos/df_imoveis_semana1.parquet')

In [39]:
from pyspark.sql.types import StringType

df_final = df_final.withColumn('caracteristicas', df_final['caracteristicas'].cast(StringType()))

df_final.write.csv('/home/luizh/Python/1Challenge/Data Science/Alura-Challenge-DS/Salvamentos/df_imoveis_semana1.csv')

In [40]:
%%time
df_parquet = spark.read.parquet('/home/luizh/Python/1Challenge/Data Science/Alura-Challenge-DS/Salvamentos/df_imoveis_semana1.parquet')
df_parquet.show(5)

+-----+----------+---------+---------+--------------------+--------------------+-------+------+------------+------------+-----------+----+---------+----------+----------+----+-----+-----+
|andar|area_total|area_util|banheiros|     caracteristicas|                  id|quartos|suites|tipo_anuncio|tipo_unidade|   tipo_uso|vaga|   bairro|      zona|condominio|iptu| tipo|valor|
+-----+----------+---------+---------+--------------------+--------------------+-------+------+------------+------------+-----------+----+---------+----------+----------+----+-----+-----+
|    3|        43|       43|        1|[Academia, Churra...|d2e3a3aa-09b5-45a...|      2|  NULL|       Usado| Apartamento|Residencial|   1|Paciência|Zona Oeste|       245|NULL|Venda|15000|
|    2|        42|       42|        1|[Churrasqueira, P...|085bab2c-87ad-452...|      2|  NULL|       Usado| Apartamento|Residencial|   1|Paciência|Zona Oeste|         0|   0|Venda|15000|
|    1|        41|       41|        1|[Portaria 24h, Co...|1

In [41]:
%%time
df_csv = spark.read.csv('/home/luizh/Python/1Challenge/Data Science/Alura-Challenge-DS/Salvamentos/df_imoveis_semana1.csv')
df_csv.show(5)

+---+---+---+---+--------------------+--------------------+---+----+-----+-----------+-----------+----+---------+----------+----+----+-----+-----+
|_c0|_c1|_c2|_c3|                 _c4|                 _c5|_c6| _c7|  _c8|        _c9|       _c10|_c11|     _c12|      _c13|_c14|_c15| _c16| _c17|
+---+---+---+---+--------------------+--------------------+---+----+-----+-----------+-----------+----+---------+----------+----+----+-----+-----+
|  3| 43| 43|  1|[Academia, Churra...|d2e3a3aa-09b5-45a...|  2|NULL|Usado|Apartamento|Residencial|   1|Paciência|Zona Oeste| 245|NULL|Venda|15000|
|  2| 42| 42|  1|[Churrasqueira, P...|085bab2c-87ad-452...|  2|NULL|Usado|Apartamento|Residencial|   1|Paciência|Zona Oeste|   0|   0|Venda|15000|
|  1| 41| 41|  1|[Portaria 24h, Co...|18d22cbe-1b86-476...|  2|NULL|Usado|Apartamento|Residencial|   1|Guaratiba|Zona Oeste|   0|   0|Venda|20000|
|  3| 43| 43|  1|[Churrasqueira, P...|bed8a354-9317-442...|  2|NULL|Usado|Apartamento|Residencial|   0|   Cosmos|Zona 

Apesar do csv ter sido mais rápido neste contexto, temos que no arquivo csv não é carregado os nomes das colunas, sendo dada a função para quem for mexer nos arquivos. Outro ponto pe que nossos dados são ainda muito pequenos comparados a grandes bancos de dados, talvez se o dataset fosse maior o parquet teria sido mais rápido.

## **Conclusão**

Para essa primeira semana fizemos os primeiros passos para preparar nosso dados para a criação de modelos de previsão e recomendação de imóveis. Comparado ao início, o DataFrame apresentou um melhora siginificativa quanto na apresentação de seus dados, estando mais limpo e com a tipagem correta de seus dados.

Para a próxima semana ainda vamos precisar tratar mais um pouco nossos dados. Como é possível ver algumas colunas apresentam valores nulos ou ainda informações relevantes de nosso dados estão dentro de uma única coluna, como é possível ver em `caracteristicas`, mas na semana que vem já criaremos nosso modelo.